In [0]:
spark

# Reading the file

In [0]:
file_path = "/Volumes/my_data_catalog/raw_zone/csv_files/customer_10mb.csv"

df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(file_path)

display(df)

In [0]:
df.printSchema()

# Cleaning


In [0]:
from pyspark.sql.functions import *

df = df.fillna({'city': 'unknown', 'state': 'unknown', 'total_money' : 'NULL'})

# Data Transformation

In [0]:
df = df.withColumn('last_active_year', year(col('last_active_day')))\
       .withColumn('last_active_month', month(col('last_active_day')))

df.show(5)

In [0]:
unique_cities = df.select(countDistinct('city')).collect()
unique_cities[0][0]

In [0]:
from pyspark.sql import functions as F

state_summary = df.groupBy('state').agg(
    F.round(F.sum("total_money"), 2).alias('state_revenue'),
    F.round(F.avg('total_transactions'), 2).alias('total_transaction_per_cust'),
    F.count('customer_id').alias('customer_count')
).orderBy(F.desc("state_revenue"))

state_summary.show()

In [0]:
from pyspark.sql.window import Window

ranked_customers = (df
    .withColumn("rank", F.rank().over(Window.partitionBy("city").orderBy(F.desc("total_money"))))
    .filter(F.col("rank") <= 3))

ranked_customers.show()

In [0]:
running_total_window = Window.orderBy("last_transaction_date")

print(type(running_total_window))

In [0]:
pivot_df = df.groupBy('state').pivot('city').sum('total_money')

pivot_df.show()

In [0]:
from pyspark.sql import functions as F

result = df.groupBy('state').agg(F.count_distinct('city').alias('city_count'))
result.show()

In [0]:
n_df = df.withColumn('days_offline', F.datediff(F.current_date(), F.col('last_active_day')))

n_df.select("name", "city", "state", "days_offline").show()